# Shard-map viewer

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/shardmap_viewer.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/shardmap_viewer.ipynb): it queries real ICESat-2 ATL06 granule **metadata** anonymously from NASA CMR-STAC (no Earthdata Login -- the CMR-STAC search is anonymous) and renders with `ipyleaflet`. The Binder image already provides `zagg[analysis,catalog,viz]` (incl. `stac-geoparquet` and `ipyleaflet`) via the repo's `.binder/` environment._

This notebook demonstrates the `zagg.viz` shard-map viewer (issue
[#38](https://github.com/englacial/zagg/issues/38)) with **real ICESat-2
ATL06 granules** fetched anonymously from NASA CMR-STAC.

The viewer has two layers:

- **Headless render core** (`zagg.viz.render_shardmap` and friends) — pure
  Python, no browser/widget stack. Turns a `ShardMap` (plus an optional
  `Catalog`) into WGS84 GeoJSON `FeatureCollection` dicts: shard outlines
  and granule footprints.
- **ipyleaflet wrapper** (`zagg.viz.show_shardmap`) — builds an interactive
  context basemap + shard layer + toggleable granule-footprint layer.

**Polar-aware projection.** `show_shardmap` picks the display CRS from the
map's extent: a polar AOI renders in NASA polar-stereographic (EPSG:3031
Antarctic / EPSG:3413 Arctic) with a **GIBS** basemap — undistorted at the
pole — while mid-latitude AOIs stay on Web Mercator + OpenStreetMap.

## Install

On Binder this is already set up by `.binder/postBuild`. Locally:

```bash
pip install "zagg[analysis,catalog,viz]"   # core + stac-geoparquet + ipyleaflet widget
```

The `viz` extra pulls in `ipyleaflet` (the interactive map); `catalog` pulls
in `stac-geoparquet` (the CMR-STAC catalog). **Requires an internet
connection** for the anonymous CMR-STAC query in section 1 — no credentials
are needed.

## Areas of interest

The notebook queries two AOIs:

1. **Antarctic Peninsula** (`[-65, -70, -55, -64]` WGS84 lon/lat) — January
   2020. Dense ICESat-2 coverage near the pole; O(10–40) granules in two
   weeks. Renders in EPSG:3031 (Antarctic Polar Stereographic).
2. **Jakobshavn Glacier, West Greenland** (`[-52, 68, -45, 72]`) — June
   2020. Second example showing the same pipeline on an Arctic AOI
   (EPSG:3413).

Both inputs (ShardMap JSON and STAC-geoparquet Catalog) are supported from
day one — see section 3 for the round-trip to disk and reload.

## 1. Antarctic Peninsula — fetch real ATL06 granules from NASA CMR-STAC

`CMRSource` speaks directly to NASA's CMR-STAC endpoint (`requests`).
No Earthdata Login or credentials are needed for anonymous granule-metadata
queries. The returned `Catalog` wraps a stac-geoparquet Arrow table (one row
per granule, both S3 and HTTPS asset hrefs preserved).

In [ ]:
from zagg.catalog.sources import CMRSource, Query

# Antarctic Peninsula AOI — two weeks in January 2020.
AOI_BBOX   = (-65.0, -70.0, -55.0, -64.0)   # lon_min, lat_min, lon_max, lat_max
START_DATE = "2020-01-01"
END_DATE   = "2020-01-15"

query = Query(
    short_name="ATL06",
    version="007",
    start_date=START_DATE,
    end_date=END_DATE,
    region=AOI_BBOX,
    provider="NSIDC_CPRD",
)

catalog = CMRSource().fetch(query)
print(f"Fetched {len(catalog)} granules for {query.collection} over {AOI_BBOX}")

In [ ]:
# Inspect the catalog schema — each row is a STAC Item with WKB geometry.
print("Schema columns:", catalog.table.schema.names[:10], "...")
print("Total rows:", catalog.table.num_rows)

# Decode a few granule records to confirm footprints are present.
records = catalog.granule_records()
print(f"\n{len(records)} records with valid footprint geometry.")
for rec in records[:3]:
    print(f"  {rec['id']}  |  https: {(rec['https'] or 'None')[:70]}...")

## 2. Build a ShardMap on a HEALPix grid

We use a HEALPix grid (`parent_order=6, child_order=12`) — the same
configuration as `src/zagg/configs/atl06.yaml`. The `mortie` backend is
used for HEALPix intersection and requires no extra install. If `spherely`
is installed, pass `backend='auto'` to prefer exact S2 intersections.

`ShardMap.build` maps each grid shard (a parent-order HEALPix cell) to the
set of granules whose footprint intersects it. The manifest is self-contained
— it stores `{"id", "s3", "https"}` per granule, so the aggregation runner
never needs the Catalog again at run time.

In [ ]:
from zagg.catalog.shardmap import ShardMap
from zagg.grids import HealpixGrid

# HEALPix grid matching atl06.yaml (parent_order=6, child_order=12, fullsphere).
grid = HealpixGrid(parent_order=6, child_order=12, layout="fullsphere")

# Build: catalog footprints are intersected with the HEALPix shard cells
# that cover the AOI. Use backend='auto' to prefer spherely when available.
shardmap = ShardMap.build(catalog, grid, backend="mortie")

print(f"ShardMap: {len(shardmap.shard_keys)} shards, "
      f"{shardmap.metadata['total_pairs']} granule-shard pairs")
print(f"Build time: {shardmap.metadata['build_wall_s']:.2f}s  "
      f"backend: {shardmap.metadata['backend']}")
print(f"Grid signature: {shardmap.grid_signature}")

In [ ]:
# Inspect a few shard assignments.
for key, shard_granules in zip(shardmap.shard_keys[:4], shardmap.granules[:4]):
    ids = [g["id"] for g in shard_granules]
    print(f"  shard {key:6d}: {len(ids)} granule(s) — {ids[:2]}{'...' if len(ids) > 2 else ''}")

## 3. Persist to disk and reload (round-trip)

Both the ShardMap JSON and the STAC-geoparquet Catalog are supported as
file-path inputs to `show_shardmap`. Here we round-trip both to disk to
demonstrate the saved-file path you would use in practice (e.g. after
running `python -m zagg.catalog --config atl06.yaml ...`).

In [ ]:
import tempfile
from pathlib import Path

tmp      = Path(tempfile.mkdtemp())
sm_path  = tmp / "shardmap_ATL06_peninsula_jan2020.json"
cat_path = tmp / "catalog_ATL06_peninsula_jan2020.parquet"

shardmap.to_json(str(sm_path))
catalog.to_geoparquet(str(cat_path))

print(f"ShardMap -> {sm_path}  ({sm_path.stat().st_size / 1024:.1f} KB)")
print(f"Catalog  -> {cat_path}  ({cat_path.stat().st_size / 1024:.1f} KB)")

# Reload to verify round-trip.
from zagg.catalog.sources import Catalog

sm_rt  = ShardMap.from_json(str(sm_path))
cat_rt = Catalog.from_geoparquet(str(cat_path))
print(f"\nRound-trip OK: {len(sm_rt.shard_keys)} shards, {len(cat_rt)} granules")

## 4. Headless render core: GeoJSON FeatureCollections

`render_shardmap` assembles every layer into one dict of GeoJSON
`FeatureCollection`s. Pass a `catalog` to include granule footprints. Each
value is plain, JSON-serializable GeoJSON — no widgets required.

In [ ]:
from zagg.viz import render_shardmap

layers = render_shardmap(shardmap, catalog)
{name: (fc and len(fc["features"])) for name, fc in layers.items()}

In [ ]:
# One feature per shard, with the shard key and its granule count.
shards_fc = layers["shards"]
feat = shards_fc["features"][0]
print("geometry type:", feat["geometry"]["type"])
print("properties:", feat["properties"])

# All populated shards in the Antarctic Peninsula window.
[f["properties"] for f in shards_fc["features"]][:5]

In [ ]:
# One feature per granule footprint, decoded from the real Catalog.
print(f"{len(layers['granules']['features'])} granule footprint(s)")
[f["properties"]["id"] for f in layers["granules"]["features"]][:5]

## 5. Interactive map — Antarctic Peninsula (ipyleaflet, polar-stereographic)

`show_shardmap` builds an `ipyleaflet.Map` from the saved ShardMap and
Catalog paths. Both ShardMap JSON and STAC-geoparquet Catalog are accepted
as file paths or in-memory objects.

**Projection is auto-selected from the map's extent.** This Antarctic AOI is
entirely south of −60°, so the viewer renders in **EPSG:3031** (Antarctic
Polar Stereographic) with a NASA **GIBS** polar basemap instead of the
distorted Web Mercator default. An Arctic AOI (poleward of +60°) gets
**EPSG:3413**; mid-latitude AOIs stay on Web Mercator + OpenStreetMap. Pass
`crs="EPSG:3857"` to force Mercator, or `crs="EPSG:3413"`/`"EPSG:3031"` to
force a specific pole. Vector layers stay WGS84 GeoJSON — proj4leaflet
reprojects them client-side.

**Run in JupyterLab** (Binder already has the deps via `.binder/postBuild`;
locally `pip install "zagg[analysis,catalog,viz]"`) to see the live map.
Under headless `nbconvert` the Map object is constructed (no error) but
tiles won't display.

### Verification checklist

1. **Basemap** — the GIBS polar basemap renders, pans, zooms, and the
   continent is undistorted at the pole (no Web-Mercator stretching).
2. **Shard outlines** — blue polygons over the Peninsula; click one for
   its `shard_key` and `n_granules` properties.
3. **Granule footprints toggle** — layer-control (top-right); the ICESat-2
   track footprints appear/disappear.

In [ ]:
from zagg.viz import show_shardmap

# File-path interface — same as after `python -m zagg.catalog ...`.
# CRS auto-selected from the AOI extent: this Antarctic map renders in EPSG:3031.
m = show_shardmap(str(sm_path), catalog=str(cat_path), zoom=5)
print("display CRS:", m.crs["name"])
print("layers:", [getattr(layer, "name", type(layer).__name__) for layer in m.layers])
m

## 6. Second AOI — Jakobshavn Glacier, West Greenland

A second example using an Arctic AOI to show the same pipeline working
independently on a different region. Jakobshavn Glacier is one of
Greenland's fastest-moving glaciers and a standard ICESat-2 study region.

In [ ]:
# render_shardmap / show_shardmap re-imported here for a standalone re-run of
# this section; CMRSource / Query / ShardMap / grid still come from sections 1-2.
from zagg.viz import render_shardmap, show_shardmap

# Second AOI: Jakobshavn Glacier, West Greenland — June 2020.
AOI2_BBOX   = (-52.0, 68.0, -45.0, 72.0)
START2, END2 = "2020-06-01", "2020-06-15"

query2 = Query(
    short_name="ATL06",
    version="007",
    start_date=START2,
    end_date=END2,
    region=AOI2_BBOX,
    provider="NSIDC_CPRD",
)

catalog2  = CMRSource().fetch(query2)
shardmap2 = ShardMap.build(catalog2, grid, backend="mortie")

print(f"Greenland AOI: {len(catalog2)} granules, "
      f"{len(shardmap2.shard_keys)} shards")

# Headless render — confirm layers are populated.
layers2 = render_shardmap(shardmap2, catalog2)
print("Layer feature counts:",
      {name: (fc and len(fc["features"])) for name, fc in layers2.items()})

In [ ]:
# Save and display interactive map for the Greenland AOI.
tmp2      = Path(tempfile.mkdtemp())
sm2_path  = tmp2 / "shardmap_ATL06_greenland_jun2020.json"
cat2_path = tmp2 / "catalog_ATL06_greenland_jun2020.parquet"

shardmap2.to_json(str(sm2_path))
catalog2.to_geoparquet(str(cat2_path))

# Arctic AOI -> auto-selects EPSG:3413 (NSIDC Sea Ice Polar Stereographic North).
m2 = show_shardmap(str(sm2_path), catalog=str(cat2_path), zoom=5)
print("display CRS:", m2.crs["name"])
m2